# PBG de Mendoza (1970-2022) — serie única, a precios constantes de 1999

Reconstruye una serie continua de PBG a partir de los tramos publicados por la DEIE, empalmados así:

- **1986-2022**: usa los años de solapamiento reales (1991-1993) entre los tramos "pesos de 1996" y "miles de pesos de 1993" (ancla en 1993).
- **1970-1985 → 1986**: no hay año de solapamiento. Se usa el **PBI nacional (Base Ferreres)** como proxy del crecimiento real de Mendoza en 1986, para despejar el coeficiente de empalme implícito. Ese coeficiente combina moneda + recálculo metodológico de la DEIE — no aísla el tipo de cambio Austral-Peso por sí solo.

Toda la serie se reexpresa a precios constantes de 1999 con el IPC INDEC.

**Fuente:** DEIE Mendoza, Facultad de Ciencias Económicas (FCE) - UNCuyo, y Olguín, P.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

## 1. Datos crudos por tramo (DEIE) + PBI nacional (Ferreres)

In [ ]:
PBG_A = {1970: 313.43, 1971: 334.41, 1972: 356.26, 1973: 364.33, 1974: 406.37, 1975: 355.2, 1976: 404.05, 1977: 380.63, 1978: 423.6, 1979: 481.05, 1980: 429.24, 1981: 351.83, 1982: 360.43, 1983: 419.26, 1984: 404.44, 1985: 384.61}   # 1970-1985, a valores constantes de 1970, en Australes
PBG_B = {1986: 267608.7, 1987: 271052.0, 1988: 264473.0, 1989: 253410.0, 1990: 253627.0, 1991: 259579.0, 1992: 277469.0, 1993: 296965.0}   # 1986-1993, en pesos de 1996 (año base 1996)
PBG_C = {1991: 6485506.946923, 1992: 7035025.586794, 1993: 7761508.224701, 1994: 8098028.310572, 1995: 7908811.385646, 1996: 8135361.877227, 1997: 8905959.444905, 1998: 9488295.932283, 1999: 9252615.169366, 2000: 9002038.451338, 2001: 8322993.49667, 2002: 7772198.039427, 2003: 9339929.594535}   # 1991-2003, en miles de pesos de 1993
PBG_D = {2004: 10874929.097084, 2005: 11380763.411128, 2006: 12259303.976934, 2007: 12770718.617713, 2008: 13065173.973136, 2009: 12700761.719855, 2010: 13262431.723353, 2011: 13739583.70382, 2012: 13545085.639385, 2013: 14211513.620518, 2014: 13687924.078648, 2015: 14189502.676874, 2016: 13373002.759949, 2017: 13655492.409271, 2018: 13585607.412423, 2019: 13410851.445972, 2020: 12342645.880974, 2021: 13635887.646933, 2022: 14178694.442282}   # 2004-2022, en miles de pesos de 1993
IPC   = {1970: 4.75e-10, 1971: 6.4e-10, 1972: 1.015e-09, 1973: 1.627e-09, 1974: 2.021e-09, 1975: 5.714e-09, 1976: 3.1086e-08, 1977: 8.5808e-08, 1978: 2.36409e-07, 1979: 6.13507e-07, 1980: 1.231702e-06, 1981: 2.51855e-06, 1982: 6.668583e-06, 1983: 2.9595142e-05, 1984: 0.000215077833, 1985: 0.0016607925, 1986: 0.003157004167, 1987: 0.007303241667, 1988: 0.03235, 1989: 1.028553583333, 1990: 24.8289, 1991: 67.4531, 1992: 84.2489, 1993: 93.189, 1994: 97.0818, 1995: 100.3594, 1996: 100.5156, 1997: 101.0469, 1998: 101.9813, 1999: 100.7915, 2000: 99.8449, 2001: 98.781666666667, 2002: 124.335, 2003: 141.05, 2004: 147.28, 2005: 167.48, 2006: 179.08, 2007: 194.89, 2008: 238.54536, 2009: 287.92424952, 2010: 334.56797794224, 2011: 427.912443788125, 2012: 527.616043190758, 2013: 650.550581254205, 2014: 844.414654467958, 2015: 1192.313492108756, 2016: 1513.045821486012, 2017: 2066.820592149892, 2018: 2546.322969528668, 2019: 3860.22562180546, 2020: 5921.586103849576, 2021: 8136.259306689318} # IPC INDEC, nivel general, base 1999=100
PBI   = {1969: 152643.7734, 1970: 160860.9564, 1971: 166912.6062, 1972: 170379.3345, 1973: 176760.9743, 1974: 186316.018, 1975: 185210.5515, 1976: 185188.5521, 1977: 197015.0269, 1978: 190666.3861, 1979: 203891.6518, 1980: 207011.4322, 1981: 195787.0856, 1982: 189602.2415, 1983: 197400.6105, 1984: 201349.0246, 1985: 187351.746, 1986: 200726.1196, 1987: 205926.3718, 1988: 202022.1639, 1989: 188010.8197, 1990: 184568.7671, 1991: 204093.8254, 1992: 223701.268, 1993: 236504.9802, 1994: 250307.8855, 1995: 243186.1015, 1996: 256626.2431, 1997: 277441.3176, 1998: 288122.8046, 1999: 278369.0139, 2000: 276172.1854, 2001: 263995.6744, 2002: 235234.5968, 2003: 256022.4652, 2004: 279020} # PBI nacional, precios de mercado, Base Ferreres (mill. $ 1993)

## 2. Empalme A→B: PBI nacional como proxy de crecimiento real en 1986

In [ ]:
g_1986 = PBI[1986] / PBI[1985] - 1
print(f"Crecimiento real PBI nacional 1985->1986: {g_1986*100:.4f}%")

K = PBG_B[1986] / (PBG_A[1985] * (1 + g_1986))
print(f"K (moneda + rebase, implicito): {K:.6f}")

## 3. Empalme B→C: años de solapamiento reales (1991-1993), ancla 1993

In [ ]:
coef_B = PBG_C[1993] / PBG_B[1993]
print(f"Coeficiente de empalme B->C (ancla 1993): {coef_B:.4f}")

factor_1999 = IPC[1999] / IPC[1993]
print(f"Factor de reexpresion a precios de 1999: {factor_1999:.4f}")

## 4. Serie única, 1970-2022, en miles de $ constantes de 1999

In [ ]:
serie_1993 = {}
for y, v in PBG_A.items():
    if y <= 1985:
        serie_1993[y] = v * K * coef_B
for y, v in PBG_B.items():
    if y <= 1990:
        serie_1993[y] = v * coef_B
for y, v in PBG_C.items():
    serie_1993[y] = v
for y, v in PBG_D.items():
    serie_1993[y] = v

serie_1999 = {y: v * factor_1999 for y, v in serie_1993.items()}

df = (pd.DataFrame(sorted(serie_1999.items()), columns=["anio", "pbg_miles_1999"])
        .assign(var_pct=lambda d: d["pbg_miles_1999"].pct_change() * 100))
df

In [ ]:
df.to_csv("pbg_mendoza_precios_1999.csv", index=False)

## 5. Gráfico

In [ ]:
navy, teal, lightgrey = "#1B2A4A", "#1F8A8C", "#D9D9D9"

fig, ax = plt.subplots(figsize=(11, 6), dpi=150)
ax.plot(df["anio"], df["pbg_miles_1999"], color=navy, linewidth=2.2)

for y in [1986, 2004]:
    ax.axvline(y, color=teal, linestyle="--", linewidth=1, alpha=0.5)

ax.set_title("Evolución del PBG de Mendoza (1970-2022)\na precios constantes de 1999",
             fontsize=14, fontweight="bold", color=navy, loc="left")
ax.set_ylabel("Miles de $ constantes de 1999", fontsize=10, color="#333333")
ax.set_xlabel("Año", fontsize=10, color="#333333")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:,.0f}M"))
ax.set_xlim(1969, 2023)
ax.grid(axis="y", color=lightgrey, linewidth=0.7)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#888888")
ax.tick_params(colors="#555555", labelsize=9)

fig.text(0.01, -0.04,
         "Fuente: DEIE Mendoza, Facultad de Ciencias Económicas (FCE) - UNCuyo, y Olguín, P.\n"
         "PBG empalmado: 1986-2022 con años de solapamiento reales; 1970-1985 empalmado usando el PBI\n"
         "nacional (Ferreres) como proxy de crecimiento real en 1986. Reexpresado a precios de 1999 con el IPC INDEC.",
         fontsize=7, color="#777777")

plt.tight_layout()
plt.savefig("pbg_mendoza_1999.png", bbox_inches="tight")
plt.show()

## 6. Descargar resultados (Colab)

In [ ]:
from google.colab import files
files.download("pbg_mendoza_precios_1999.csv")
files.download("pbg_mendoza_1999.png")